### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [18]:
### Read all the pdf files inside the directory
def process_all_pdfs(pdf_directory):
    """process all pdf in the directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory} to process.")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}...")
        try:
            # Load the PDF file
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
                doc.metadata["file_path"] = str(pdf_file)
                doc.metadata["file_size"] = os.path.getsize(pdf_file)
                doc.metadata["file_modified_time"] = os.path.getmtime(pdf_file)
                doc.metadata["file_created_time"] = os.path.getctime(pdf_file)
                doc.metadata["file_accessed_time"] = os.path.getatime(pdf_file)
                doc.metadata["file_owner"] = os.stat(pdf_file).st_uid
                doc.metadata["file_permissions"] = oct(os.stat(pdf_file).st_mode)[-3:]
                doc.metadata["file_is_readable"] = os.access(pdf_file, os.R_OK)
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file.name}.")
            
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
            
    print(f"\nFinished processing all PDFs. Total documents loaded: {len(all_documents)}")
    return all_documents

#Process all pdf files in the directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files in ../data to process.

Processing FellowShift_Backend_Test_Report_2026-11-08.pdf...
Loaded 4 documents from FellowShift_Backend_Test_Report_2026-11-08.pdf.

Processing FellowShift_Progress_and_E2E_Test_Report_2026-11-08.pdf...
Loaded 5 documents from FellowShift_Progress_and_E2E_Test_Report_2026-11-08.pdf.

Processing FINALQUEBANKJ2EEpdf__2026_10_16_14_24_58.pdf...
Loaded 1 documents from FINALQUEBANKJ2EEpdf__2026_10_16_14_24_58.pdf.

Processing J2EE_Important_Answers.pdf...
Loaded 20 documents from J2EE_Important_Answers.pdf.

Processing QuestionBankdJ2EEpdf__2026_10_13_17_23_09.pdf...
Loaded 1 documents from QuestionBankdJ2EEpdf__2026_10_13_17_23_09.pdf.

Finished processing all PDFs. Total documents loaded: 31


In [19]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T07:27:54+00:00', 'author': 'FellowShift Project Team', 'keywords': '', 'moddate': '2026-11-08T07:27:54+00:00', 'subject': 'FastAPI and Mistral production API validation', 'title': 'FellowShift Backend Verification Test Report', 'trapped': '/False', 'source': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_type': 'pdf', 'file_path': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_size': 37744, 'file_modified_time': 1794122898.0884235, 'file_created_time': 1794474450.2528963, 'file_accessed_time': 1794491953.6785011, 'file_owner': 0, 'file_permissions': '666', 'file_is_readable': True}, page_content='Internal testing record \x7f Synthetic documents only\nPage 1\nFELLOWSHIFT\nBackend Verification\nTest Repo

In [20]:
## Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks. For better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    
    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Show first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [21]:
chunks = split_documents(all_pdf_documents)
chunks

Split 31 documents into 81 chunks.

Example chunk:
Content: Internal testing record  Synthetic documents only
Page 1
FELLOWSHIFT
Backend Verification
Test Report
Production API validation  OCR extraction  Decision logic
Report date
08 November 2026
Environm...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T07:27:54+00:00', 'author': 'FellowShift Project Team', 'keywords': '', 'moddate': '2026-11-08T07:27:54+00:00', 'subject': 'FastAPI and Mistral production API validation', 'title': 'FellowShift Backend Verification Test Report', 'trapped': '/False', 'source': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_type': 'pdf', 'file_path': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_size': 37744, 'file_modified_time': 1794122898.0884235, 'file_created_time':

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T07:27:54+00:00', 'author': 'FellowShift Project Team', 'keywords': '', 'moddate': '2026-11-08T07:27:54+00:00', 'subject': 'FastAPI and Mistral production API validation', 'title': 'FellowShift Backend Verification Test Report', 'trapped': '/False', 'source': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_type': 'pdf', 'file_path': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_size': 37744, 'file_modified_time': 1794122898.0884235, 'file_created_time': 1794474450.2528963, 'file_accessed_time': 1794491953.6785011, 'file_owner': 0, 'file_permissions': '666', 'file_is_readable': True}, page_content='Internal testing record \x7f Synthetic documents only\nPage 1\nFELLOWSHIFT\nBackend Verification\nTest Repo

### Embedding and VectoreStoreDB

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager 
        
        Args:
            model_name (str): Hugging Face model name for sentence embeddings. Default is "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        print(f"Initialized embedding model: {model_name}")
        self.model = None  # Model will be loaded lazily
        self._load_model()
        
    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
            texts (List[str]): A list of text strings to embedding.

        Returns:
            np.ndarray: An array of embeddings with shape (len(texts), embedding_dim).
        """
        
        if not self.model:
            raise ValueError("Embedding model is not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self) -> int:
    #     """Get the dimension of the embeddings generated by the model
        
    #     Returns:
    #         int: The embedding dimension.
    #     """
    #     if not self.model:
    #         raise ValueError("Embedding model is not loaded. Call _load_model() first.")
        
    #     return self.model.get_sentence_embedding_dimension()
    
    ## intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Initialized embedding model: all-MiniLM-L6-v2
Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3363.49it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Dell\AppData\Local\Temp\ipykernel_9576\2557938647.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())


## VectoreStore

In [39]:

class VectorStore:
    """Manages a vector store for document embeddings using ChromaDB"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of PDF document embeddings for RAG."
                }
            )

            print(
                f"ChromaDB collection '{self.collection_name}' "
                f"initialized at '{self.persist_directory}'."
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add documents and their embeddings to the Vector Store."""

        if len(documents) != embeddings.shape[0]:
            raise ValueError(
                "Number of documents and embeddings must match."
            )

        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, doc in enumerate(documents):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(embeddings[i].tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )

            print(
                f"Successfully added {len(documents)} documents "
                "to the vector store."
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to ChromaDB: {e}")
            raise


# Initialize vector store
vector_store = VectorStore()
vector_store


ChromaDB collection 'pdf_documents' initialized at '../data/vector_store'.
Existing documents in collection: 0


In [33]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T07:27:54+00:00', 'author': 'FellowShift Project Team', 'keywords': '', 'moddate': '2026-11-08T07:27:54+00:00', 'subject': 'FastAPI and Mistral production API validation', 'title': 'FellowShift Backend Verification Test Report', 'trapped': '/False', 'source': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_type': 'pdf', 'file_path': '..\\data\\pdf\\FellowShift_Backend_Test_Report_2026-11-08.pdf', 'file_size': 37744, 'file_modified_time': 1794122898.0884235, 'file_created_time': 1794474450.2528963, 'file_accessed_time': 1794491953.6785011, 'file_owner': 0, 'file_permissions': '666', 'file_is_readable': True}, page_content='Internal testing record \x7f Synthetic documents only\nPage 1\nFELLOWSHIFT\nBackend Verification\nTest Repo

In [40]:
### convert the text to embeddings
texts = [chunk.page_content for chunk in chunks]

## Generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 81 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.11s/it]

Generated embeddings with shape: (81, 384)
Adding 81 documents to the vector store...
Successfully added 81 documents to the vector store.
Total documents in collection: 81


## Retriever Pipeline From VectorStore

In [46]:
class RAGRetriever:
    """Handles query-based retrival from the vector srote"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retirever with a vector store and embedding manager.
        
        Args:
            vector_store (VectorStore): The vector store instance for document retrieval.
            embedding_manager (EmbeddingManager): The embedding manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents based on the query.
        
        Args:
            query (str): The input query string.
            top_k (int): The number of top documents to retrieve. Default is 5.
            score_threshold (float): Minimum similarity score to consider a document relevant. Default is 0.0.
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"top_k: {top_k}, score_threshold: {score_threshold}")
        
        #Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        #Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                    
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                    else:
                        print(f"Document {doc_id} filtered out due to low similarity score: {similarity_score:.4f}")
                        
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved from the vector store.")
                
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [47]:
rag_retriever

In [49]:
rag_retriever.retrieve("What is FellowShift Backend?")

Retrieving documents for query: 'What is FellowShift Backend?'
top_k: 5, score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.53it/s]

Generated embeddings with shape: (1, 384)
Document doc_c9734e67_22 filtered out due to low similarity score: -0.0995
Document doc_3afdcaef_13 filtered out due to low similarity score: -0.1085
Document doc_4e0c5675_1 filtered out due to low similarity score: -0.1736
Retrieved 2 documents after applying score threshold.


[{'id': 'doc_84054b41_12',
  'content': 'FellowShift | Internal Development & Testing Report\nPage 1\n FELLOWSHIFT\n Progress & End-to-End Integration Test\n Report\n SIH Problem Statement 239 | Scholarship Application Platform\n Report date: 8 November 2026\n Environment tested: Local Next.js frontend + remote Supabase + production Render AI backend\nOverall result\nThe student submission-to-AI-verification flow completed successfully. Three documents reached OCR, provider\nfallback worked after a Mistral rate-limit response, and the final result was correctly persisted and shown to the\nstudent as requiring additional human review.\nThis report consolidates the development work, backend verification, frontend workflow changes, configuration tests, and the\nlatest synthetic end-to-end application test discussed in this conversation.',
  'metadata': {'page_label': '1',
   'file_modified_time': 1794162206.3575697,
   'subject': 'SIH 239 scholarship platform development and integration t

In [53]:
rag_retriever.retrieve("5. ORM – Concept, Architecture and Working 10 Marks")

Retrieving documents for query: '5. ORM – Concept, Architecture and Working 10 Marks'
top_k: 5, score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.40it/s]

Generated embeddings with shape: (1, 384)
Document doc_fb311e7f_57 filtered out due to low similarity score: -0.0177
Document doc_75fccd4b_52 filtered out due to low similarity score: -0.3167
Document doc_5a2f3daa_45 filtered out due to low similarity score: -0.3411
Document doc_071e12d8_58 filtered out due to low similarity score: -0.3427
Retrieved 1 documents after applying score threshold.


[{'id': 'doc_354856f1_51',
  'content': 'J2EE (2BCA501) - Important Answers\nPage 10\n5. ORM – Concept, Architecture and Working\n 10 Marks\nORM (Object-Relational Mapping) is a technique that connects the object-oriented model used in Java with\nthe relational model used by databases. It automatically converts Java objects to table rows and back, so the\nprogrammer works with objects instead of writing SQL and JDBC code. Hibernate is the most popular ORM\ntool for Java.\nWhy ORM? (object-relational mismatch)\nObject world (Java)\nRelational world (Database)\nClass\nTable\nObject / instance\nRow (record)\nAttribute / property\nColumn\nObject reference / collection\nForeign key / join table\nInheritance, polymorphism\nNo direct equivalent\nArchitecture of ORM\n Fig. 1 ORM sits between the Java objects and the RDBMS\nThe ORM layer (Hibernate) receives Java objects from the application, uses the mapping information,\ngenerates the proper SQL and sends it to the RDBMS through JDBC. Data co

### Get all model list

In [61]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
whisper-large-v3
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
qwen/qwen3.8-27b


## Integration VectorDB Context Pipeline With LLM Output

In [72]:
## Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize Groq LLM with API key from environment variable
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it in the .env file.")

llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=500)
#models = ["openai/gpt-oss-20b", "openai/gpt-oss-7b", "openai/gpt-oss-3b", "openai/gpt-oss-1b", "openai/gpt-oss-350m", "openai/gpt-oss-125m"]
# Test API
# response = llm.invoke("Hello! Introduce yourself in 2 sentences.")

# print(response.content)

## 2. Simple RAG Function: retrive context + generate response
def rag_simple(query, retriver, llm, top_k=5):
    results = retriver.retrieve(query, top_k=top_k)
    context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(results)])
    if not context:
        return "No relevant documents found for the query."
    
    ## Generate answer using LLM
    prompt = f"""Answer the following question based on the provided context:\n\n
        Context:\n{context}\n\n
        Question: {query}\n\n
        Answer:
        """
        
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [98]:
answer = rag_simple("What is Fellowshift Backend?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is Fellowshift Backend?'
top_k: 5, score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.14it/s]

Generated embeddings with shape: (1, 384)
Document doc_c9734e67_22 filtered out due to low similarity score: -0.0995
Document doc_3afdcaef_13 filtered out due to low similarity score: -0.1085
Document doc_4e0c5675_1 filtered out due to low similarity score: -0.1736
Retrieved 2 documents after applying score threshold.


**Fellowshift Backend** is the server‑side component of the Fellowshift scholarship‑application platform.  
It runs on a production Render AI environment and is responsible for:

* Receiving student document uploads from the Next.js front‑end.  
* Sending those documents to an AI‑based OCR/verification service (via Render AI).  
* Persisting the verification results and any required human‑review flags in a Supabase database.  
* Exposing APIs that the front‑end calls to submit documents, poll status, and retrieve results.  

In short, the Fellowshift Backend orchestrates the AI verification workflow, stores outcomes, and provides the data layer that the front‑end consumes.


## Enhanced RAG Pipeline Features

In [115]:
# Enhanced RAG Pipeline Features
def reg_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    - Return answers, sources, confidence scores, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant documents found for the query.",
            "sources": [],
            "confidence": [],
            "context": "" if return_context else None
        }
        
    # Prepare context and sources
    context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(results)])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page': doc['metadata'].get('page', 'Unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + "..." 
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)
    
    # Generate answer using LLM
    prompt = f"""Answer the following question based on the provided context:\n\n
        Context:\n{context}\n\n
        question: {query}\n\n
        Answer:
        """
    response = llm.invoke([prompt])
    
    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": [doc['similarity_score'] for doc in results]
    }
    if return_context:
        output["context"] = context
        
    return output

# Example usage of the advanced RAG pipeline
result = reg_advanced("What is FellowShift Backend?", rag_retriever, llm, top_k=3, min_score=-1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence Scores:", result["confidence"])
print("Context Preview:", result["context"][:500] + "...")

Retrieving documents for query: 'What is FellowShift Backend?'
top_k: 3, score_threshold: -1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold.


Answer: **FellowShift Backend** is the server‑side component of the FellowShift scholarship‑application platform.  
It receives student submissions from the Next.js front‑end, runs OCR on the uploaded documents, dispatches the extracted data to AI verification providers (e.g., Mistral, Gemini), handles provider‑rate‑limit or outage fallbacks, persists the verification results in a database (Supabase), and exposes a verification HTTP endpoint that returns a status (e.g., NEEDS_ATTENTION) back to the front‑end for display to the student. The backend is written in TypeScript, follows the platform’s policy rules for document selection and eligibility, and is currently tested with synthetic documents in an internal development environment.
Sources: [{'source': 'FellowShift_Progress_and_E2E_Test_Report_2026-11-08.pdf', 'page': 0, 'score': 0.054053306579589844, 'preview': 'FellowShift | Internal Development & Testing Report\nPage 1\n FELLOWSHIFT\n Progress & End-to-End Integration Test\n Repo

In [116]:
queries = [
    "What is FellowShift Backend?",
    "Which AI providers are used in FellowShift?",
    "How does document verification work in FellowShift?"
]

for query in queries:
    print("\n" + "=" * 70)
    print("QUESTION:", query)

    result = reg_advanced(
        query,
        rag_retriever,
        llm,
        top_k=3,
        min_score=-1,
        return_context=False
    )

    print("ANSWER:", result["answer"])
    print("SOURCES:", result["sources"])


QUESTION: What is FellowShift Backend?
Retrieving documents for query: 'What is FellowShift Backend?'
top_k: 3, score_threshold: -1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold.


ANSWER: **FellowShift Backend** is the server‑side component of the FellowShift scholarship‑application platform.  
It orchestrates the entire end‑to‑end workflow that a student experiences:  

* **Document ingestion** – receives PDF/scan uploads from the Next.js front‑end.  
* **OCR & extraction** – runs OCR on the documents, extracts key fields, and persists the results in Supabase.  
* **AI verification** – dispatches the extracted data to an AI provider (Mistral, Gemini, etc.), handles rate‑limit or outage fallbacks, and records the AI’s decision.  
* **Policy & mismatch checks** – applies eligibility rules, detects category mismatches, and flags documents that need human review.  
* **Status management** – updates the student’s application status (e.g., NEEDS_ATTENTION) and returns a 200 HTTP response to the front‑end.  

The backend is built with TypeScript, runs on Render, and is tightly coupled to the front‑end and Supabase database. It is the core engine that turns a student’s

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold.


ANSWER: The system uses two AI providers:

1. **Mistral** – the primary LLM that is called for document verification.  
2. **Gemini** – a fallback LLM that is invoked when Mistral returns a rate‑limit or other error.
SOURCES: [{'source': 'FellowShift_Progress_and_E2E_Test_Report_2026-11-08.pdf', 'page': 0, 'score': 0.061785995960235596, 'preview': 'FellowShift | Internal Development & Testing Report\nPage 1\n FELLOWSHIFT\n Progress & End-to-End Integration Test\n Report\n SIH Problem Statement 239 | Scholarship Application Platform\n Report date: 8 November 2026\n Environment tested: Local Next.js frontend + remote Supabase + production Render AI bac...'}, {'source': 'FellowShift_Backend_Test_Report_2026-11-08.pdf', 'page': 3, 'score': -0.08760809898376465, 'preview': 'FELLOWSHIFT  |  BACKEND VERIFICATION REPORT\nInternal testing record \x7f Synthetic documents only\nPage 4\n\x7f Provider failure path not exercised: no forced timeout or provider outage test was run.\n\x7f Frontend not 

Batches: 100%|██████████| 1/1 [00:00<00:00, 61.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold.


ANSWER: **Document verification in FellowShift is a multi‑step, backend‑driven workflow that turns a student’s uploaded paperwork into a machine‑readable record, checks it against the scholarship criteria, and flags any items that need human review. The process is orchestrated by the FellowShift Python FastAPI service and relies on an AI‑powered OCR provider (currently Mistral). Below is a high‑level overview of how the system works, with key technical details pulled from the internal verification report.**

---

### 1. **Submission**

| Step | What happens | Key details |
|------|--------------|-------------|
| **Student uploads documents** | The student uses the FellowShift front‑end to attach PDFs, images, or scans of required documents (e.g., income certificates, transcripts). | Only synthetic documents were used in the test run; real student documents are yet to be tested. |
| **POST request to the backend** | The front‑end sends a `multipart/form‑data` request to `https://fellows